<style>
h1 {
    background-color: #CADCFC;
    color: #00246B;
    font-weight: bold;
    text-align: center;
}
</style>
<h1>
    Loading Libraries 
</h1>

In [23]:
from pymongo import MongoClient
import pandas as pd
from datetime import datetime
import os

<style>
h1 {
    background-color: #CADCFC;
    color: #00246B;
    font-weight: bold;
    text-align: center;
}
</style>
<h1>
    Connecting to
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/9/93/MongoDB_Logo.svg/1200px-MongoDB_Logo.svg.png" style="height: 1em">
</h1>

In [24]:
# Connecting to DB
connection_string = open(os.path.join("..", "mongo_db_connection_string.txt"), "r").read()
client = MongoClient(connection_string)

# Test the connection
db = client["WeatherDB"]
print("Connected without errors")

Connected without errors


<style>
h1 {
    background-color: #CADCFC;
    color: #00246B;
    font-weight: bold;
    text-align: center;
}
</style>
<h1>
    Loading and filtering weather data
</h1>

In [25]:
# Set date range to our date range of interest
start_date = datetime(2023, 12, 30)
end_date = datetime(2024, 12, 30)

# Load Weather Data
weather_data = pd.read_csv(os.path.join(
    "..",
    "raw_data",
    "OpenweatherAPI_WeatherData_AllCities_2023-12-30_0000_to_2024-12-30_23-59.csv"
))
weather_data["Timestamp"] = pd.to_datetime(weather_data["Timestamp"])
filtered_weather_data = weather_data[
    (weather_data["Timestamp"] >= start_date) & (weather_data["Timestamp"] <= end_date) # Filter by date
]

# Load Air Pollution Data
air_pollution_data = pd.read_csv(os.path.join("..", "raw_data", "OpenweatherAPI_AirPollutionData_2019_to_2024.csv"))
air_pollution_data["Timestamp"] = pd.to_datetime(air_pollution_data["Timestamp"])
filtered_air_pollution_data = air_pollution_data[
    (air_pollution_data["Timestamp"] >= start_date) & (air_pollution_data["Timestamp"] <= end_date) # Filter by date
]

# Dict of solar data files for each location 
SOLCAST_DATA_PATH = os.path.join("..", "raw_data", "solcast")
solar_files = {
    solcast_file.split("_")[1].split(" (")[0]: os.path.join(SOLCAST_DATA_PATH, solcast_file)
    for solcast_file in os.listdir(SOLCAST_DATA_PATH)
}

# Loop through each file and load data
solar_dataframes = []
for location, file_path in solar_files.items():
    solar_data = pd.read_csv(file_path)
    solar_data["Location"] = location # Adding location column
    solar_dataframes.append(solar_data)

# Combine all DataFrames
combined_solar_data = pd.concat(solar_dataframes, ignore_index = True)

# Filter the solar data for the date range
combined_solar_data["period_end"] = pd.to_datetime(combined_solar_data["period_end"]).dt.tz_convert(None)
filtered_solar_data = combined_solar_data[
    (combined_solar_data["period_end"] >= start_date) & (combined_solar_data["period_end"] <= end_date) # Filter by data
]

# Merge Weather and Air data 
weather_air_data = pd.merge(
    filtered_weather_data,
    filtered_air_pollution_data,
    on = ["Timestamp", "Location"],
    how = "inner" 
)

# Rename mismatched column name to help merge 
filtered_solar_data.rename(columns = {"period_end": "Timestamp"}, inplace = True)

# Merge with solar data
all_data = pd.merge(
    weather_air_data,
    filtered_solar_data,
    on = ["Timestamp", "Location"],
    how = "inner"
)

# Removing unnecessary columns
all_data.drop(
    ["Sea Level Pressure (hPa)", "Ground Level Pressure (hPa)", "Rain (3h, mm)", "Snow (3h, mm)", "period"],
    axis = 1, 
    inplace = True
)


/var/folders/zv/nhtt3w6s7wx_2h90bptpfxp80000gn/T/ipykernel_92708/162081691.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_solar_data.rename(columns = {"period_end": "Timestamp"}, inplace = True)


Here we began preparations to upload the data to our MongoDB database. 
Firstly, we defined the dates for data to be included. Due to limitations from scraping, we only had comprehensive data from the dates 30/12/2023 to 30/12/2024, therefore, we limited all data to these dates as it still gave us a whole years worth to analyze. 

To load Weather and air pollution was relatively simple as all locations were stored together; we simply read the csv as a pandas data frame and filtered to fit our time frame using the datetime function. For the solar section we had to use a for loop to combine all locations after reading them as data frames. We also had to remove the timezone built into the data as it wasn't compatible and didn't impact our results due to all information being collected in the UK. 

We then combined all three data frames using the time and location to create a single DF containing all information.

Finally, we filtered out all columns that wouldn't be used in our DB for cleanliness and efficiency going forward

<style>
    .multi-line {
        background-color: #CADCFC;
    }
    h1, h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
    }
</style>
<div class="multi-line">
    <h1>
        Formatting weather data for MongoDB
    </h1>
    <h2>
        Creating time series collection
    </h2>
</div>



In [ ]:
# Creating collection in MongoDB
db.create_collection(
    "WeatherDB",
    timeseries={
        "timeField": "Timestamp",
        "metaField": "Location",
        "granularity": "hours"
    }
)

We used a time time series collection for its reduced complexity and increased efficiency. 

To do this we had to outline the Time, metadata, and measurements before uploading our data. For us, this meant setting time as the 'Timestamp' value; The metafield as Location as that's the primary identifier of our data; The measurement to hours as thats the frequency of the data from scraping. 

<style>
    h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h2>
    Formatting and inserting weather data into MongoDB
</h2>

In [56]:
# Transform each row into a dictionary
def transform_row(row):
    return {
        "Timestamp": row["Timestamp"],
        "Location": row["Location"],
        "Temperature": {
            "Air_Temperature": row.get("Temperature (K)", None),
            "Feels_Like_Temperature": row.get("Feels Like (K)", None),
            "Temp_Max": row.get("Temp Max (K)", None),
            "Temp_Min": row.get("Temp Min (K)", None),
        },
        "Humidity": row.get("Humidity (%)", None),
        "Wind": {
            "Wind_Speed": row.get("Wind Speed (m/s)", None),
            "Wind_Direction": row.get("Wind Direction (°)", None),
        },
        "Cloudiness": row.get("Cloudiness (%)", None),
        "Weather_Info": {
            "Weather_Main": row.get("Weather Main", None),
            "Weather_Description": row.get("Weather Description", None),
            "Weather_Icon": row.get("Weather Icon", None),
        },
        "Rain": {
            "Rain_1h_mm": row.get("Rain (1h, mm)", None),
        },
        "Snow": {
            "Snow_1h_mm": row.get("Snow (1h, mm)", None),
        },
        "Air_Pollution": {
            "AQI": row.get("AQI", None),
            "CO": row.get("CO (μg/m³)", None),
            "NO": row.get("NO (μg/m³)", None),
            "NO2": row.get("NO2 (μg/m³)", None),
            "O3": row.get("O3 (μg/m³)", None),
            "SO2": row.get("SO2 (μg/m³)", None),
            "PM2.5": row.get("PM2.5 (μg/m³)", None),
            "PM10": row.get("PM10 (μg/m³)", None),
            "NH3": row.get("NH3 (μg/m³)", None),
        },
        "Solar_Irradiation": {
            "dni": row.get("dni", None),
            "ghi": row.get("ghi", None),
        },
    }

# Apply transformation
documents = all_data.apply(transform_row, axis = 1).tolist()

# Upload to MongoDB
db.WeatherDB.insert_many(documents)

InsertManyResult([ObjectId('6782d67d9fec7f8c8625339c'), ObjectId('6782d67d9fec7f8c8625339d'), ObjectId('6782d67d9fec7f8c8625339e'), ObjectId('6782d67d9fec7f8c8625339f'), ObjectId('6782d67d9fec7f8c862533a0'), ObjectId('6782d67d9fec7f8c862533a1'), ObjectId('6782d67d9fec7f8c862533a2'), ObjectId('6782d67d9fec7f8c862533a3'), ObjectId('6782d67d9fec7f8c862533a4'), ObjectId('6782d67d9fec7f8c862533a5'), ObjectId('6782d67d9fec7f8c862533a6'), ObjectId('6782d67d9fec7f8c862533a7'), ObjectId('6782d67d9fec7f8c862533a8'), ObjectId('6782d67d9fec7f8c862533a9'), ObjectId('6782d67d9fec7f8c862533aa'), ObjectId('6782d67d9fec7f8c862533ab'), ObjectId('6782d67d9fec7f8c862533ac'), ObjectId('6782d67d9fec7f8c862533ad'), ObjectId('6782d67d9fec7f8c862533ae'), ObjectId('6782d67d9fec7f8c862533af'), ObjectId('6782d67d9fec7f8c862533b0'), ObjectId('6782d67d9fec7f8c862533b1'), ObjectId('6782d67d9fec7f8c862533b2'), ObjectId('6782d67d9fec7f8c862533b3'), ObjectId('6782d67d9fec7f8c862533b4'), ObjectId('6782d67d9fec7f8c862533

Here, we're transforming each row into a formatted dictionary so it can be uploaded to MongoDB as a document. Each row represents a 'time point' at a specific location so our results are a compilation of documents with json style formatting that are identified by a time and region. 
We then upload them to our database using the insert_many() function for increased efficiency. 

The result is a collection of documents, "WeatherDB", which are hierachally structured and have metadata containing information on the weather conditions. 

<style>
    h1 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h1>
    Loading, Preparing, and uploading Farm & Crop data
</h1>

For the farm and crop data, it wasn't related to time so we could upload each as individual clusters. This would help with querying by creating distinct locations for each, making it more concisely stored and more efficient to query. 

In [4]:
# Loading CSVs
crops_info = pd.read_csv('raw_data/Crops_Info.csv')
farms_info = pd.read_csv('raw_data/Farms_Info.csv')

# Converting to dictionary formatting
crops_data = crops_info.to_dict(orient="records")
farms_data = farms_info.to_dict(orient="records")

# Creating crops collection and uploading data
db.create_collection("crops")
db.crops.insert_many(crops_data)

# Creating Farms collection and uploading data
db.create_collection("farms")
db.farms.insert_many(farms_data)

InsertManyResult([ObjectId('678c6574d4417c8637b37cc9'), ObjectId('678c6574d4417c8637b37cca'), ObjectId('678c6574d4417c8637b37ccb'), ObjectId('678c6574d4417c8637b37ccc'), ObjectId('678c6574d4417c8637b37ccd'), ObjectId('678c6574d4417c8637b37cce'), ObjectId('678c6574d4417c8637b37ccf'), ObjectId('678c6574d4417c8637b37cd0'), ObjectId('678c6574d4417c8637b37cd1'), ObjectId('678c6574d4417c8637b37cd2'), ObjectId('678c6574d4417c8637b37cd3'), ObjectId('678c6574d4417c8637b37cd4'), ObjectId('678c6574d4417c8637b37cd5'), ObjectId('678c6574d4417c8637b37cd6'), ObjectId('678c6574d4417c8637b37cd7'), ObjectId('678c6574d4417c8637b37cd8'), ObjectId('678c6574d4417c8637b37cd9'), ObjectId('678c6574d4417c8637b37cda'), ObjectId('678c6574d4417c8637b37cdb'), ObjectId('678c6574d4417c8637b37cdc'), ObjectId('678c6574d4417c8637b37cdd'), ObjectId('678c6574d4417c8637b37cde'), ObjectId('678c6574d4417c8637b37cdf'), ObjectId('678c6574d4417c8637b37ce0'), ObjectId('678c6574d4417c8637b37ce1'), ObjectId('678c6574d4417c8637b37c

This process was much more simple and involved: loading the CSVs from our 'raw_data' file; Converting them to dictionary style format; Creating the new respective collections and uploading the data. 

This new data could be joined to our existing collection through the respective foreign keys, thus allowing weather information relating to farms and crops to be called and vice versa.